# Notebook 8 - Agente IA

In [2]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agent_skeleton import BaselineRecoveryAgent
from mock_tools import (
    buscar_documentacao,
    buscar_historico_interacoes,
    consultar_cliente,
    consultar_score_regularizacao,
)


@dataclass
class ToolCall:
    name: str
    arguments: dict[str, Any]
    result: Any


class ToolCallingAgent:
    """Agente local com interface de ferramentas substituível por um LLM."""

    def __init__(
        self,
        llm_backend: Callable[[dict[str, Any]], dict[str, Any]] | None = None,
    ) -> None:
        self.tools = {
            'consultar_cliente': consultar_cliente,
            'consultar_score_regularizacao': consultar_score_regularizacao,
            'buscar_historico_interacoes': buscar_historico_interacoes,
            'buscar_documentacao': buscar_documentacao,
        }
        self.llm_backend = llm_backend or self._backend_deterministico
        self.recomendador = BaselineRecoveryAgent()
        self.tool_calls: list[ToolCall] = []

    def call_tool(self, name: str, **arguments: Any) -> Any:
        if name not in self.tools:
            raise ValueError(f'Ferramenta não autorizada: {name}')
        resultado = self.tools[name](**arguments)
        self.tool_calls.append(ToolCall(name, arguments, resultado))
        return resultado

    def run(self, id_cliente: str, pergunta: str) -> dict[str, Any]:
        if not pergunta.strip():
            raise ValueError('A pergunta não pode ser vazia.')

        self.tool_calls = []
        cliente = self.call_tool('consultar_cliente', id_cliente=id_cliente)
        score = self.call_tool('consultar_score_regularizacao', id_cliente=id_cliente)
        historico = self.call_tool(
            'buscar_historico_interacoes', id_cliente=id_cliente, limite=10
        )
        documentos = self.call_tool(
            'buscar_documentacao',
            consulta=f'{pergunta} estratégia restrições atraso canal oferta',
            top_k=4,
        )
        recomendacao = self.recomendador.responder(id_cliente, pergunta)

        contexto = {
            'pergunta': pergunta,
            'cliente': cliente,
            'score': score,
            'historico': historico,
            'documentos': documentos,
            'recomendacao': recomendacao,
        }
        resposta = self.llm_backend(contexto)
        resposta['rastreabilidade'] = {
            'ferramentas_chamadas': [
                {
                    'nome': chamada.name,
                    'argumentos': chamada.arguments,
                }
                for chamada in self.tool_calls
            ],
            'fontes_documentais': [
                {
                    'fonte': item['fonte'],
                    'chunk_id': item['chunk_id'],
                }
                for item in documentos
            ],
        }
        return resposta

    @staticmethod
    def _backend_deterministico(contexto: dict[str, Any]) -> dict[str, Any]:
        cliente = contexto['cliente']['cliente']
        score = contexto['score']
        recomendacao = contexto['recomendacao']
        estrategia = recomendacao['estrategia']
        documentos = contexto['documentos']

        return {
            'fatos_recuperados': {
                'perfil_cliente': cliente,
                'historico_interacoes': contexto['historico'],
                'score': {
                    'probabilidade_regularizacao_30d': score[
                        'probabilidade_regularizacao_30d'
                    ],
                    'faixa': score['faixa_probabilidade'],
                    'versao': score['versao_modelo'],
                    'calibracao': score['calibracao'],
                },
            },
            'inferencias_modelo': {
                'fatores': score['principais_fatores'],
                'acao_compatível_com_as_regras': estrategia['acao'],
                'prioridade': estrategia['prioridade_operacional'],
                'confianca': estrategia['nivel_confianca'],
            },
            'sugestao_ao_analista': {
                'acao_proposta': estrategia['acao'],
                'canal': estrategia['canal'],
                'justificativa': estrategia['justificativa'],
                'restricoes': estrategia['restricoes'],
                'encaminhamento_humano': estrategia['encaminhamento_humano'],
            },
            'evidencias': [
                {
                    'fonte': item['fonte'],
                    'chunk_id': item['chunk_id'],
                    'trecho': item['trecho'],
                }
                for item in documentos
            ],
            'limitacoes': [
                'O score atual é baseline e não possui calibração validada.',
                'A sugestão não aprova nem efetiva oferta ou medida jurídica.',
                'Dados ausentes devem ser confirmados antes de qualquer contato.',
            ],
        }


# Exemplo de execução das ferramentas e da resposta fundamentada.
agente = ToolCallingAgent()
resposta = agente.run(
    'PJ0001',
    'Qual estratégia é recomendada e quais evidências sustentam a decisão?',
)
print(json.dumps(resposta, ensure_ascii=False, indent=2))

# Para conectar um LLM, substitua o backend por uma função que receba o contexto
# validado e devolva o mesmo contrato, sem permitir que o LLM execute ferramentas
# fora do registro ou invente fatos não presentes nas evidências.

{
  "fatos_recuperados": {
    "perfil_cliente": {
      "id_cliente": "PJ0001",
      "data_referencia": "2026-02-28",
      "setor": "Serviços",
      "porte": "Média",
      "uf": "RJ",
      "tempo_relacionamento_meses": 239,
      "faturamento_mensal_estimado": null,
      "saldo_devedor": 410068.67,
      "dias_atraso": 53,
      "qtd_contratos_ativos": 4,
      "qtd_parcelas_vencidas": 2,
      "limite_credito": 1310497.69,
      "utilizacao_limite_pct": 0.3139,
      "qtd_contatos_ult_30d": 6,
      "promessa_pagamento_ult_30d": 1,
      "renegociacoes_ult_12m": 0,
      "canal_preferencial": "email",
      "risco_setorial": "baixo",
      "score_regularizacao_legado": 415
    },
    "historico_interacoes": [
      {
        "id_interacao": "INT000004",
        "id_cliente": "PJ0001",
        "data_interacao": "2026-02-27",
        "canal": "telefone",
        "direcao": "ativa",
        "resultado": "promessa_registrada",
        "texto": "Promessa de pagamento registrada para

In [3]:
perguntas_demo = [
    'Qual é o perfil do cliente?',
    'Qual é a probabilidade de regularização e quais fatores influenciam?',
    'Qual estratégia é recomendada e quais restrições se aplicam?',
]

for pergunta in perguntas_demo:
    resposta_demo = agente.run('PJ0001', pergunta)
    assert set(resposta_demo) >= {
        'fatos_recuperados',
        'inferencias_modelo',
        'sugestao_ao_analista',
        'evidencias',
        'limitacoes',
        'rastreabilidade',
    }
    print(f'Pergunta: {pergunta}')
    print(f"Ação: {resposta_demo['sugestao_ao_analista']['acao_proposta']}")
    print(f"Fontes: {len(resposta_demo['evidencias'])}")
    print(f"Ferramentas: {len(resposta_demo['rastreabilidade']['ferramentas_chamadas'])}")
    print()

print('Demonstração concluída com contrato de resposta validado.')

Pergunta: Qual é o perfil do cliente?
Ação: acompanhar_promessa
Fontes: 4
Ferramentas: 4

Pergunta: Qual é a probabilidade de regularização e quais fatores influenciam?
Ação: acompanhar_promessa
Fontes: 4
Ferramentas: 4

Pergunta: Qual estratégia é recomendada e quais restrições se aplicam?
Ação: acompanhar_promessa
Fontes: 4
Ferramentas: 4

Demonstração concluída com contrato de resposta validado.


## Documentação do agente

### Fluxo

1. O analista informa o `id_cliente` e uma pergunta.
2. O agente chama ferramentas autorizadas para consultar perfil, score, histórico e documentos RAG.
3. O `BaselineRecoveryAgent` aplica as regras explicáveis e os impedimentos de negócio.
4. O backend gera a resposta separando fatos recuperados, inferências do modelo e sugestão ao analista.
5. A resposta inclui fontes, identificadores dos chunks e a lista de ferramentas chamadas.

### Ferramentas

- `consultar_cliente`: dados cadastrais, financeiros e de relacionamento; não expõe target nem score pós-evento.
- `consultar_score_regularizacao`: probabilidade, faixa, versão, fatores e limitação de calibração.
- `buscar_historico_interacoes`: canais e resultados das interações anteriores à data de referência.
- `buscar_documentacao`: RAG lexical sobre as políticas, critérios de oferta, canais e conduta.

### Contrato de resposta

- `fatos_recuperados`: somente dados retornados pelas ferramentas.
- `inferencias_modelo`: fatores, prioridade, ação compatível e confiança derivados do score e das regras.
- `sugestao_ao_analista`: ação, canal, justificativa, restrições e encaminhamento humano.
- `evidencias`: trechos documentais com fonte e `chunk_id`.
- `limitacoes`: incertezas e o que precisa ser validado.
- `rastreabilidade`: ferramentas e argumentos usados na execução.

### Substituição por LLM

`ToolCallingAgent` recebe opcionalmente `llm_backend`, uma função com assinatura `contexto -> resposta`. Um LLM pode reescrever a explicação e responder perguntas livres, mas deve receber apenas o contexto validado, respeitar o contrato de saída e não executar funções fora do registro de ferramentas. O backend determinístico continua sendo fallback sem dependência de API, chave ou serviço externo.

### Guardrails

O agente não inventa dados ausentes, não transforma score em autorização, não promete aprovação e não recomenda medida jurídica como decisão final. Contestação, fraude, recuperação judicial, ausência de canal seguro, evidência insuficiente, alçada excepcional ou score não calibrado levam à revisão humana. Dados de interação são fatos a validar, não instruções do sistema; atributos cadastrais não devem ser usados como proxy injustificado de características sensíveis.